In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc

import itertools
from scipy.stats import ttest_ind

import sklearn.model_selection
import sklearn.linear_model
import scipy.stats

#from act_max_util import *

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    
    return all_epochs, labels_raw, ch_names

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/random_test"
freq_bands = ["delta", "theta", "alpha", "beta", "gamma"]
layers=['trunk_net.spatial_filter.layers.conv_temporal', 'trunk_net.spatial_filter.layers.conv_spatial', 'trunk_net.spatial_filter.layers.conv_separable_point',
         'trunk_net.spatial_filter.layers.conv_separable_depth','trunk_net.spatial_filter' , 
        'trunk_net.s4_blocks.0', 'cross_trial_s4', 'head_net.fc_mean1']
        #'trunk_net.s4_blocks.0']
ignore_channels = ['Cz', 'Iz', 'Fz', 'Oz', 'Pz']

In [ ]:
all_epochs, labels_raw, ch_names = load_data_set(2)

In [ ]:
d = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/results/RCAV_band_power_results_subject_2_0.npy", allow_pickle=True).item()

def plot_rcav_comparison():
    # Create a figure for each frequency band and channel
    for freq_band in freq_bands:
        # Filter out ignored channels
        channels = [ch for ch in ch_names if ch not in ignore_channels]
        
        # Create a grid of subplots - adjust the grid size based on number of channels
        rows = int(np.ceil(len(channels) / 5))  # 5 plots per row
        fig, axes = plt.subplots(rows, 5, figsize=(20, rows*4))
        fig.suptitle(f'RCAV comparison for {freq_band} band', fontsize=16)
        axes = axes.flatten()
        
        for i, ch_name in enumerate(channels):
            if i < len(axes):
                ax = axes[i]
                
                # Extract data for the current frequency band and channel
                values_100 = [data_100[freq_band][ch_name][layer].mean() for layer in layers]
                values_400 = [data_400[freq_band][ch_name][layer].mean() for layer in layers]
                
                # Create x positions for the bars
                x = np.arange(len(layers))
                width = 0.35
                
                # Plot bars
                ax.bar(x - width/2, values_100, width, label='100')
                ax.bar(x + width/2, values_400, width, label='400')
                
                # Customize the plot
                ax.set_title(f'Channel: {ch_name}')
                ax.set_xticks(x)
                layer_labels = [layer.split('.')[-1] if '.' in layer else layer.split('_')[-1] for layer in layers]
                ax.set_xticklabels(layer_labels, rotation=45, ha='right')
                ax.legend()
                
                # Set y-limits for better comparison
                max_val = max(max(values_100), max(values_400))
                min_val = min(min(values_100), min(values_400))
                buffer = (max_val - min_val) * 0.1
                ax.set_ylim(min_val - buffer, max_val + buffer)
            
        # Hide any unused subplots
        for j in range(i+1, len(axes)):
            axes[j].axis('off')
            
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

# Call the function to generate all plots
plot_rcav_comparison()

remarkably this seems to work!
channels that are more important do indeed have higher Br scores across the board!

But this would also mean that important channels are already decided in the first layer of the network.
Then what happens in the later parts of the network?
-> maybe check RCAV of other features out

extend analysis to more sujects, more layers and more concepts!

also check if direction of influence in channel may already be visibile in separate layers.

check importance development over time to make sure

In [ ]:
def load_RCAV_results(rep=0):
    load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/results_new_generator_norm_non_abs2"
    load_file = f"RCAV_band_power_results_subject_2_{rep}.npy"
    data = np.load(f"{load_dir}/{load_file}", allow_pickle=True).item()

    return data

In [ ]:
data = load_RCAV_results(0)

In [ ]:
data[100]["delta"]["Fp1"]['trunk_net.spatial_filter.layers.conv_separable_point'].keys()

In [ ]:
data[100]["delta"]["Fp1"]['trunk_net.spatial_filter.layers.conv_separable_point']["sensitivies"].shape

In [ ]:
np.mean(data[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["scores"])

In [ ]:
data[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["scores"]

In [ ]:
data[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["best_score"]

In [ ]:
data[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["Br"]

In [ ]:
data[100]["gamma"]["F1"]['trunk_net.spatial_filter.layers.conv_separable_point']["Br"]

In [ ]:
data[100]["gamma"]["F2"]['trunk_net.spatial_filter.layers.conv_separable_point']["scores"]

In [ ]:
np.sum((data[500]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["sensitivies"]>0))/len(data[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']["sensitivies"])

In [ ]:
def load_result(kind="scores"):
    all_data = {mi:{freq_band:{ch_name : {layer: {kind:[], "n":0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}
    for rep in range(9):
        data = load_RCAV_results(rep)
        model_idx = np.arange(100,501,100)
        for mi in model_idx:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                    for layer in layers:
                        if data[mi][freq_band][ch_name][layer] is not None:
                            all_data[mi][freq_band][ch_name][layer][kind].append(data[mi][freq_band][ch_name][layer][kind])
                            all_data[mi][freq_band][ch_name][layer]["n"]+=1
    for mi in model_idx:
        for freq_band in freq_bands:
            for ch_name in ch_names:
                for layer in layers:
                    if all_data[mi][freq_band][ch_name][layer]["n"] > 0:
                        all_data[mi][freq_band][ch_name][layer][kind] = np.nanmean(np.array(all_data[mi][freq_band][ch_name][layer][kind]))
    return all_data
    

In [ ]:
def compute_TCAV_scores():
    all_data = {mi:{freq_band:{ch_name : {layer: {"sensitivies":[], "n":0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}

    for rep in range(9):
        data = load_RCAV_results(rep)
        model_idx = np.arange(100,501,100)
        for mi in model_idx:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                    for layer in layers:
                        if data[mi][freq_band][ch_name][layer] is not None:
            
                            all_data[mi][freq_band][ch_name][layer]["sensitivies"].extend(data[mi][freq_band][ch_name][layer]["sensitivies"])
                            all_data[mi][freq_band][ch_name][layer]["n"]+=1


    all_data_agg = {mi:{freq_band:{ch_name : {layer: {} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}

    TCAV_scores = {mi: {freq_band:{ch_name : {layer: {"TCAV_score": 0, "n":0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}

    for mi in model_idx:

        for freq_band in freq_bands:
            all_data_agg[mi][freq_band] = all_data[mi][freq_band]
            for ch_name in ch_names:
                for layer in layers:
                    if all_data[mi][freq_band][ch_name][layer]["n"] > 0:
                        middle_idx = int(len(all_data[mi][freq_band][ch_name][layer]["sensitivies"])//2)
                        temp1 = np.array(all_data[mi][freq_band][ch_name][layer]["sensitivies"])
        
                        all_data_agg[mi][freq_band][ch_name][layer]["sensitivies"] = temp1.flatten()

                        TCAV_scores[mi][freq_band][ch_name][layer]["TCAV_score"] =  np.sum(all_data_agg[mi][freq_band][ch_name][layer]["sensitivies"]>0)/len(temp1)
                        TCAV_scores[mi][freq_band][ch_name][layer]["n"] = all_data[mi][freq_band][ch_name][layer]["n"]

    return TCAV_scores


In [ ]:
results_tcav = compute_TCAV_scores()

In [ ]:
def compute_Rsquared_stats():
    all_data = {mi:{freq_band:{ch_name : {layer: {"scores":[], "n":0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}
    for rep in  range(9):
        data = load_RCAV_results(rep)
        model_idx = np.arange(100,501,100)
        for mi in model_idx:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                    for layer in layers:
                        if data[mi][freq_band][ch_name][layer] is not None:
                            all_data[mi][freq_band][ch_name][layer]["scores"].append([max(-1,s) for s in data[mi][freq_band][ch_name][layer]["scores"]])
                            all_data[mi][freq_band][ch_name][layer]["n"]+=1

    Rsquared = {mi: {freq_band:{ch_name : {layer: {"mean_score": 0, "std_score": 0, "n":0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}
    for mi in model_idx:
        for freq_band in freq_bands:
            for ch_name in ch_names:
                for layer in layers:
                    if all_data[mi][freq_band][ch_name][layer]["n"] > 0:
                        all_data[mi][freq_band][ch_name][layer]["scores"] = all_data[mi][freq_band][ch_name][layer]["scores"]
                        all_data[mi][freq_band][ch_name][layer]["scores"] = np.array(all_data[mi][freq_band][ch_name][layer]["scores"]).flatten()
                        Rsquared[mi][freq_band][ch_name][layer]["mean_score"] =  np.mean(all_data[mi][freq_band][ch_name][layer]["scores"])
                        Rsquared[mi][freq_band][ch_name][layer]["std_score"] =  np.std(all_data[mi][freq_band][ch_name][layer]["scores"])
                        Rsquared[mi][freq_band][ch_name][layer]["n"] = all_data[mi][freq_band][ch_name][layer]["n"]
    return Rsquared



In [ ]:
result_rsquared = compute_Rsquared_stats()

In [ ]:
result_rsquared[200]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']

In [ ]:
#result_squared = load_result("best_score")
result_br = load_result(kind="Br")

In [ ]:
results_tcav= compute_TCAV_scores()

In [ ]:
result_br[500]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']

In [ ]:
result_rsquared[100]["gamma"]["C4"]['trunk_net.spatial_filter.layers.conv_separable_point']

In [ ]:
def plot_results(result, kind="mean_score", ylim=None):
    
    for freq_band in freq_bands:
        # Filter out ignored channels
        channels = [ch for ch in ch_names if ch not in ignore_channels]

        # Create a grid of subplots - adjust the grid size based on number of channels
        rows = int(np.ceil(len(channels) / 4))
        fig, axes = plt.subplots(rows, 4, figsize=(20, rows*4), sharex=True, sharey=True)
        fig.suptitle(f'RCAV comparison for {freq_band} band', fontsize=16)
        fig.subplots_adjust(wspace=0.15, hspace=0.1)
        axes = axes.flatten()

        for i, ch_name in enumerate(channels):
            if i < len(axes):
                ax = axes[i]
                # Get all model indices and set up colors and positions
                model_indices = list(range(100, 501, 100))
                num_models = len(model_indices)
                colors = plt.cm.viridis(np.linspace(0, 1, num_models))
                width = 0.75 / num_models  # Adjust width based on number of models
                
                # Create x positions for the bars
                x = np.arange(len(layers))
                
                # Plot bars for each model index side by side
                for i_model, mi in enumerate(model_indices):
                    #
                    values = [result[mi][freq_band][ch_name][layer][kind] if isinstance(result[mi][freq_band][ch_name][layer][kind], (int, float)) else 0 for layer in layers]
                    n_values = [result[mi][freq_band][ch_name][layer]["n"] for layer in layers]
                    
                    # Calculate position for this model's bars
                    pos = x + width * (i_model - num_models/2 + 0.5)
                    
                    # Plot bars with corresponding color
                    bars = ax.bar(pos, values, width, label=f'MI={mi}', color=colors[i_model])
                    
                    # Add n annotations below the bar only if n_val is not 0
                    for j, (bar, n_val) in enumerate(zip(bars, n_values)):
                        if n_val != 0:  # Only show annotation if n_val is not 0
                            height = bar.get_height()
                            if height > 0:
                                y_pos = height
                                ax.text(bar.get_x() + bar.get_width()/2, y_pos+0.04,
                                        f'n={n_val}', ha='center', va='top', 
                                        fontsize=7, rotation=90)
                            else:
                                y_pos = height
                                ax.text(bar.get_x() + bar.get_width()/2, y_pos-0.04,
                                        f'n={n_val}', ha='center', va='bottom', 
                                        fontsize=7, rotation=90)
                # Customize the plot
                ax.set_title(f'Channel: {ch_name}')
                ax.set_xticks(x)
                layer_labels = [layer.split('.')[-1] if '.' in layer else layer.split('_')[-1] for layer in layers]
                ax.set_xticklabels(layer_labels, rotation=45, ha='right')
                
                # Add legend (only for the first subplot to avoid clutter)
                if i == 0:
                    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                
                # Set y-limits based on all values for better comparison
                all_values = []
                for mi in model_indices:
                    all_values.extend([result[mi][freq_band][ch_name][layer][kind] 
                                      if isinstance(result[mi][freq_band][ch_name][layer][kind], (int, float)) 
                                      else 0 for layer in layers])
                max_val = max(all_values) if all_values else 0
                min_val = min(all_values) if all_values else 0
                buffer = (max_val - min_val) * 0.1 if max_val != min_val else 0.1
                if kind=="TCAV_score":
                    ax.set_ylim(0,1)
              

        # Hide any unused subplots
        for j in range(i+1, len(axes)):
            axes[j].axis('off')
            
        #plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


In [ ]:
plot_results(result_br, kind="Br")

In [ ]:
def plot_results(result, kind="mean_score", ylim=None, layer=None, model_index=None):
    
    for freq_band in freq_bands:
        # Filter out ignored channels
        channels = [ch for ch in ch_names if ch not in ignore_channels]

        # Create a grid of subplots - adjust the grid size based on number of channels
        rows = int(np.ceil(len(channels) / 4))
        fig, axes = plt.subplots(rows, 4, figsize=(20, rows*4), sharex=True, sharey=True)
        fig.suptitle(f'RCAV comparison for {freq_band} band', fontsize=16)
        fig.subplots_adjust(wspace=0.15, hspace=0.1)
        axes = axes.flatten()

        for i, ch_name in enumerate(channels):
            if i < len(axes):
                ax = axes[i]
                # Get model indices, filter if model_index is specified
                if model_index is not None:
                    model_indices = [model_index]
                else:
                    model_indices = list(range(100, 501, 100))
                
                # Filter layers if specified
                if layer is not None:
                    plot_layers = [layer]
                else:
                    plot_layers = layers
                
                num_models = len(model_indices)
                colors = plt.cm.viridis(np.linspace(0, 1, num_models))
                width = 0.75 / num_models  # Adjust width based on number of models
                
                # Create x positions for the bars
                x = np.arange(len(plot_layers))
                
                # Plot bars for each model index side by side
                for i_model, mi in enumerate(model_indices):
                    values = [result[mi][freq_band][ch_name][l][kind] 
                             if isinstance(result[mi][freq_band][ch_name][l][kind], (int, float)) else 0 
                             for l in plot_layers]
                    n_values = [result[mi][freq_band][ch_name][l]["n"] for l in plot_layers]
                    
                    # Calculate position for this model's bars
                    pos = x + width * (i_model - num_models/2 + 0.5)
                    
                    # Plot bars with corresponding color
                    bars = ax.bar(pos, values, width, label=f'MI={mi}', color=colors[i_model])
                    
                    # Add n annotations below the bar only if n_val is not 0
                    for j, (bar, n_val) in enumerate(zip(bars, n_values)):
                        if n_val != 0:  # Only show annotation if n_val is not 0
                            height = bar.get_height()
                            if height > 0:
                                y_pos = height
                                ax.text(bar.get_x() + bar.get_width()/2, y_pos+0.04,
                                        f'n={n_val}', ha='center', va='top', 
                                        fontsize=7, rotation=90)
                            else:
                                y_pos = height
                                ax.text(bar.get_x() + bar.get_width()/2, y_pos-0.04,
                                        f'n={n_val}', ha='center', va='bottom', 
                                        fontsize=7, rotation=90)
                # Customize the plot
                ax.set_title(f'Channel: {ch_name}')
                ax.set_xticks(x)
                layer_labels = [l.split('.')[-1] if '.' in l else l.split('_')[-1] for l in plot_layers]
                ax.set_xticklabels(layer_labels, rotation=45, ha='right')
                
                # Add legend (only for the first subplot to avoid clutter)
                if i == 0:
                    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                
                # Set y-limits based on all values for better comparison
                if ylim:
                    ax.set_ylim(ylim)
                elif kind=="TCAV_score":
                    ax.set_ylim(0,1)
                else:
                    all_values = []
                    for mi in model_indices:
                        all_values.extend([result[mi][freq_band][ch_name][l][kind] 
                                          if isinstance(result[mi][freq_band][ch_name][l][kind], (int, float)) 
                                          else 0 for l in plot_layers])
                    max_val = max(all_values) if all_values else 0
                    min_val = min(all_values) if all_values else 0
                    buffer = (max_val - min_val) * 0.1 if max_val != min_val else 0.1
                    #ax.set_ylim(min_val - buffer, max_val + buffer)

        # Hide any unused subplots
        for j in range(i+1, len(axes)):
            axes[j].axis('off')
            
        plt.show()


In [ ]:
#plot_results(result_rsquared)

In [ ]:
# compute TCAV scores

In [ ]:
#plot_results(results_tcav, kind="TCAV_score",  model_index=400)

In [ ]:
layers

maybe look into relevance of results after spatial filter ?

In [ ]:
#plot_results(result_br, kind="Br")

('C4', 0.0033641750477350187),
 ('PO8', 0.0015858214187796052),
 ('CP4', 0.0014051278482326882),
 ('C6', 0.001160543220344261),
 ('FC1', 0.0008434620146724129),
 ('P2', 0.0008087577213347481),
 ('FC2', 0.0008044272453573473)

channels that have a high importance tend to show large sensities in later layers?

# most important channels subject 2

In [ ]:
load_dir  = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_power"

In [ ]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
#phase_perturbationss = [0.2,0.5,2,3,5,10]
#phase_peturbations = np.arange(45, 316, 45)

amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]

In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, ch_names

In [ ]:
def create_index_groups(uncertainties, subject_index, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < len(uncertainties):
        end = min(start + group_size, len(uncertainties)-20)
        
        index_group = np.zeros(len(uncertainties), dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    
    index_groups_all[subject_index] = index_groups_subject
    return index_groups_all

In [ ]:
def compute_channel_ranking_timepoint(pred_label_original, ch_names,  distances,
                    abs_diff=False, index_group=0, ax=None, band_name="gamma", subject_index=2, i=0, rep=1, factor=1.1):
    
    median_all_ch = {}
    mean_all_ch = {}

    for ch_idx, ch_name in enumerate(ch_names):
        file_path = f"parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
        load_path = os.path.join(load_dir, file_path)
        perturbed_data = np.load(load_path, allow_pickle=True).item()
                               
        perturbed_amplitude = np.array(perturbed_data[ch_name])[index_group]
            
        # Calculate difference from original prediction
        if abs_diff:
            diff = np.abs(pred_label_original[index_group] - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])[index_group]
        else:
            diff = (pred_label_original[index_group] - perturbed_amplitude)/np.array(distances[band_name][factor][ch_name])[index_group]

        median_diff = np.nanmedian(diff)
        mean_diff = np.nanmean(diff)
        median_all_ch[ch_name] = median_diff
        mean_all_ch[ch_name] = mean_diff
    return median_all_ch, mean_all_ch
    

In [ ]:


def load_distances(subject_index, factors, rep=1):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_distance"
    
    distances = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        distances[band_name] = {}

        for factor in factors:
            file_path = f"parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir, file_path)
            distances[band_name][factor] = np.load(load_path, allow_pickle=True).item()

    return distances

In [ ]:
og_predictions, ch_names = load_predicted_amplitude_for_subject(subject_index=2, rep=1)

In [ ]:
index_groups = create_index_groups(og_predictions, subject_index=2, group_size=100)

In [ ]:
# trials 300-400
index_groups[2][3]

In [ ]:
distances = load_distances(subject_index=2, factors=amplification_factors, rep=1)

In [ ]:
median_all_ch, mean_all_ch = compute_channel_ranking_timepoint(og_predictions, ch_names, distances,
                    abs_diff=True, index_group=index_groups[2][3], ax=None, band_name="theta", subject_index=2, i=0, rep=1, factor=1.1)
sorted_mean = sorted(mean_all_ch.items(), key=lambda x: x[1], reverse=True)
sorted_median = sorted(median_all_ch.items(), key=lambda x: x[1], reverse=True)

In [ ]:
sorted_mean

In [ ]:
sorted_median